In [37]:
input_geotiff = 'confirmed_products/los_angeles_fires_2025/11SLT/OPERA_L3_DIST-ALERT-S1_T11SLT_20250303T140056Z_20251016T224900Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T11SLT_20250303T140056Z_20251016T224900Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'
rgb_tiff = 'rgb.tif'

In [38]:
import rasterio
import numpy as np

def indexed_to_rgb_robust(input_path, output_path):
    with rasterio.open(input_path) as src:
        colormap = src.colormap(1)
        if colormap is None:
            return

        indexed_data = src.read(1)

        max_index = max(colormap.keys())
        colormap_array = np.zeros((max_index + 1, 4), dtype=np.uint8)

        for index, rgba in colormap.items():
            colormap_array[index] = rgba

        rgb_rgba_data = colormap_array[indexed_data]

        rgb_data = rgb_rgba_data[:, :, :3].transpose(2, 0, 1)

        profile = src.profile
        profile.update(
            dtype=rasterio.uint8, 
            count=3,           
            nodata=None        
        )

        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(rgb_data, indexes=[1, 2, 3])
            print(f"Successfully converted and saved RGB file to {output_path}")

In [39]:
indexed_to_rgb_robust(input_geotiff, rgb_tiff)

Successfully converted and saved RGB file to rgb.tif


In [42]:
!rio pmtiles {rgb_tiff} los_angeles_2025.pmtiles --format PNG --resampling nearest --zoom-levels 5..16

100%|████████████████████████████████████| 63396/63396 [01:34<00:00, 669.36it/s]


In [43]:
import geopandas as gpd

In [44]:
df = gpd.read_parquet('/Users/cmarshak/bekaert-team/dist-s1-events/db/event_perimeters/los_angeles_fires_2025.parquet')
df.to_file('los_angeles_fires.geojson', driver='GeoJSON')